In [1]:
%pip install pandas scikit-learn matplotlib ucimlrepo


  Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.28.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.8.0-py3-none-any.whl.metadata (29 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/10.9 MB ? eta -:--:--
    --------------------------------------- 0.3/10.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.9 MB 1.8 MB/s eta 0:00:06
   --- ------------------------------------ 1.0/10.9 MB 1.9 MB/s eta 0:00:06
   ------ --------------------------------- 1.8/10.9 MB 2.3 MB/s eta 0:00:04
   --------- ------------------------------ 2.6/10.9 MB 2.7 MB/s eta 0:00:04
   ------------- -------------------------- 3.7/10.9 MB 3.1 MB/s eta 0:00:03
   ------------------ --------------------- 5.0/10.9 MB 3.6 M

## 1단계: 데이터 불러오기 (Statlog German Credit)

UCI 머신러닝 저장소의 `ucimlrepo` 패키지로 **German Credit** 데이터(id=144)를 가져온다.

- `fetch_ucirepo(id=144)`가 돌려주는 객체 안에
  - `data.features` → 신청자 정보 20개 컬럼 (X)
  - `data.targets` → 정답 컬럼 `class` (y)
  - `variables` → 각 컬럼의 설명표 (다음 단계에서 이름 바꿀 때 사용)
- 확인할 것: 행이 **1,000개**, 속성이 **20개**인지, 그리고 `class` 값이 몇 개씩 있는지 (good 700 / bad 300 이어야 함).

> 처음 실행할 때는 인터넷에서 내려받으니 몇 초 걸릴 수 있다.

In [1]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

german = fetch_ucirepo(id=144)

X = german.data.features   # 신청자 정보 (20개 컬럼)
y = german.data.targets    # 정답 (class 컬럼 1개)

print("X 크기:", X.shape)   # (1000, 20) 이어야 함
print("y 크기:", y.shape)   # (1000, 1) 이어야 함
print()
print("class 값별 인원수:")
print(y["class"].value_counts())

X.head()

X 크기: (1000, 20)
y 크기: (1000, 1)

class 값별 인원수:
class
1    700
2    300
Name: count, dtype: int64


,Attribute1,Attribute2,Attribute3,Attribute4,Attribute5,Attribute6,Attribute7,Attribute8,Attribute9,Attribute10,Attribute11,Attribute12,Attribute13,Attribute14,Attribute15,Attribute16,Attribute17,Attribute18,Attribute19,Attribute20
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,4,A121,67,A143,A152,2,A173,1,A192,A201
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,2,A121,22,A143,A152,1,A173,1,A191,A201
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,3,A121,49,A143,A152,1,A172,2,A191,A201
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,4,A122,45,A143,A153,1,A173,2,A191,A201
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,4,A124,53,A143,A153,2,A173,2,A191,A201


## 2-1단계: 컬럼 이름을 읽기 쉽게 바꾸기

`Attribute1`, `Attribute2` 같은 이름은 무슨 뜻인지 알 수 없다. 
`german.variables` 설명표의 `description`을 보고, 각 컬럼에 짧은 영어 이름을 붙인다.

- 딕셔너리 `{옛 이름: 새 이름}`을 만들고 `X.rename(columns=...)`로 바꾼다.
- 가장 중요한 것: **`Attribute13` → `age`** (나이, 단위: 년). 이번 공정성 감사의 핵심 컬럼이다.
- 마지막에 설명표와 새 이름을 나란히 출력해서 하나씩 맞게 붙었는지 눈으로 확인한다.

In [4]:
new_names = {
    "Attribute1":  "checking_status",          # 당좌예금 계좌 상태
    "Attribute2":  "duration_months",          # 대출 기간 (개월)
    "Attribute3":  "credit_history",           # 과거 신용 기록
    "Attribute4":  "purpose",                  # 대출 목적
    "Attribute5":  "credit_amount",            # 대출 금액 (DM, 독일 마르크)
    "Attribute6":  "savings",                  # 저축 계좌 / 채권
    "Attribute7":  "employment_since",         # 현재 직장 근무 기간
    "Attribute8":  "installment_rate",         # 가처분소득 대비 할부금 비율 (1~4)
    "Attribute9":  "personal_status_sex",      # 결혼 상태 + 성별
    "Attribute10": "other_debtors",            # 공동 채무자 / 보증인
    "Attribute11": "residence_since",          # 현재 집 거주 기간 (1~4)
    "Attribute12": "property",                 # 보유 재산
    "Attribute13": "age",                      # 나이 (년)  ← 감사 대상
    "Attribute14": "other_installment_plans",  # 다른 할부 계획
    "Attribute15": "housing",                  # 주거 형태
    "Attribute16": "existing_credits",         # 이 은행의 기존 대출 수
    "Attribute17": "job",                      # 직업
    "Attribute18": "num_dependents",           # 부양 책임 인원 수
    "Attribute19": "telephone",                # 전화 등록 여부
    "Attribute20": "foreign_worker",           # 외국인 노동자 여부
}

X = X.rename(columns=new_names)

# 확인: 원래 설명과 새 이름을 나란히 놓고 보기
check = german.variables[["name", "description", "type"]].copy()
check["new_name"] = check["name"].map(new_names)
check

,name,description,type,new_name
0,Attribute1,Status of existing checking account,Categorical,checking_status
1,Attribute2,Duration,Integer,duration_months
2,Attribute3,Credit history,Categorical,credit_history
3,Attribute4,Purpose,Categorical,purpose
4,Attribute5,Credit amount,Integer,credit_amount
5,Attribute6,Savings account/bonds,Categorical,savings
6,Attribute7,Present employment since,Categorical,employment_since
7,Attribute8,Installment rate in percentage of disposable i...,Integer,installment_rate
8,Attribute9,Personal status and sex,Categorical,personal_status_sex
9,Attribute10,Other debtors / guarantors,Categorical,other_debtors


## 2-2단계: 코드값(A11 등)의 뜻을 표로 정리

범주형 컬럼의 값은 `A11`, `A34`처럼 코드로 되어 있다. 코드의 뜻은 데이터 안에 없고, UCI 데이터 페이지의 원본 설명 문서(`german.doc`)에만 있다. 그래서 그 문서를 보고 직접 표로 옮긴다.

- 각 줄 = (컬럼, 코드, 원문 설명 `meaning_en`, 한국어 뜻)
- 금액 단위 **DM** = 독일 마르크 (1994년에 UCI에 공개된 독일 은행 자료)
- `count` 컬럼: 실제 데이터에서 그 코드를 가진 사람이 몇 명인지 센 값. **0이면 문서에는 있지만 데이터에는 없는 코드**다.
  - `A47` (휴가): 문서에만 있고 데이터에는 0명
  - `A95` (미혼 여성): 문서에만 있고 데이터에는 0명 → 그래서 `personal_status_sex`로는 미혼 여성과 다른 여성을 구분할 수 없다 (발표 때 데이터 한계로 언급할 만함)
- 마지막 줄은 표에 빠진 코드가 없는지 확인한다. **"빠진 코드: 없음"이 나와야 정상.**

In [3]:
# (컬럼, 코드, 원문 설명, 한국어 뜻) — 출처: UCI german.doc
codes = [
    ("checking_status", "A11", "... < 0 DM", "잔고 0 DM 미만 (마이너스)"),
    ("checking_status", "A12", "0 <= ... < 200 DM", "잔고 0 이상 200 DM 미만"),
    ("checking_status", "A13", "... >= 200 DM / salary assignments for at least 1 year", "잔고 200 DM 이상 또는 1년 이상 급여 이체"),
    ("checking_status", "A14", "no checking account", "당좌 계좌 없음"),

    ("credit_history", "A30", "no credits taken / all credits paid back duly", "대출 없음 / 모두 제때 상환"),
    ("credit_history", "A31", "all credits at this bank paid back duly", "이 은행 대출 모두 제때 상환"),
    ("credit_history", "A32", "existing credits paid back duly till now", "현재 대출을 지금까지 제때 상환 중"),
    ("credit_history", "A33", "delay in paying off in the past", "과거 상환 연체 있음"),
    ("credit_history", "A34", "critical account / other credits existing (not at this bank)", "위험 계좌 / 다른 은행에 대출 있음"),

    ("purpose", "A40", "car (new)", "새 차"),
    ("purpose", "A41", "car (used)", "중고차"),
    ("purpose", "A42", "furniture/equipment", "가구/장비"),
    ("purpose", "A43", "radio/television", "라디오/TV"),
    ("purpose", "A44", "domestic appliances", "가전제품"),
    ("purpose", "A45", "repairs", "수리"),
    ("purpose", "A46", "education", "교육"),
    ("purpose", "A47", "(vacation - does not exist?)", "휴가 (데이터에 없음)"),
    ("purpose", "A48", "retraining", "재교육/직업훈련"),
    ("purpose", "A49", "business", "사업"),
    ("purpose", "A410", "others", "기타"),

    ("savings", "A61", "... < 100 DM", "100 DM 미만"),
    ("savings", "A62", "100 <= ... < 500 DM", "100 이상 500 DM 미만"),
    ("savings", "A63", "500 <= ... < 1000 DM", "500 이상 1000 DM 미만"),
    ("savings", "A64", "... >= 1000 DM", "1000 DM 이상"),
    ("savings", "A65", "unknown / no savings account", "모름 / 저축 계좌 없음"),

    ("employment_since", "A71", "unemployed", "실업"),
    ("employment_since", "A72", "... < 1 year", "1년 미만"),
    ("employment_since", "A73", "1 <= ... < 4 years", "1년 이상 4년 미만"),
    ("employment_since", "A74", "4 <= ... < 7 years", "4년 이상 7년 미만"),
    ("employment_since", "A75", "... >= 7 years", "7년 이상"),

    ("personal_status_sex", "A91", "male : divorced/separated", "남성: 이혼/별거"),
    ("personal_status_sex", "A92", "female : divorced/separated/married", "여성: 이혼/별거/기혼"),
    ("personal_status_sex", "A93", "male : single", "남성: 미혼"),
    ("personal_status_sex", "A94", "male : married/widowed", "남성: 기혼/사별"),
    ("personal_status_sex", "A95", "female : single", "여성: 미혼 (데이터에 없음)"),

    ("other_debtors", "A101", "none", "없음"),
    ("other_debtors", "A102", "co-applicant", "공동 신청인"),
    ("other_debtors", "A103", "guarantor", "보증인"),

    ("property", "A121", "real estate", "부동산"),
    ("property", "A122", "building society savings agreement / life insurance", "주택저축 / 생명보험"),
    ("property", "A123", "car or other", "자동차 또는 기타"),
    ("property", "A124", "unknown / no property", "모름 / 재산 없음"),

    ("other_installment_plans", "A141", "bank", "은행"),
    ("other_installment_plans", "A142", "stores", "상점"),
    ("other_installment_plans", "A143", "none", "없음"),

    ("housing", "A151", "rent", "월세/전세"),
    ("housing", "A152", "own", "자가"),
    ("housing", "A153", "for free", "무상 거주"),

    ("job", "A171", "unemployed / unskilled - non-resident", "실업 / 비숙련 - 비거주자"),
    ("job", "A172", "unskilled - resident", "비숙련 - 거주자"),
    ("job", "A173", "skilled employee / official", "숙련 직원 / 공무원"),
    ("job", "A174", "management / self-employed / highly qualified employee / officer", "관리직 / 자영업 / 고급 전문직"),

    ("telephone", "A191", "none", "없음"),
    ("telephone", "A192", "yes, registered under the customer's name", "있음 (본인 명의)"),

    ("foreign_worker", "A201", "yes", "외국인 노동자 예"),
    ("foreign_worker", "A202", "no", "외국인 노동자 아님"),
]

code_table = pd.DataFrame(codes, columns=["column", "code", "meaning_en", "뜻"])

# 실제 데이터에서 각 코드가 몇 명인지 세기
code_table["count"] = [(X[col] == code).sum() for col, code in zip(code_table["column"], code_table["code"])]

# 확인: 데이터에는 있는데 표에 빠진 코드가 있는지
missing = []
for col in code_table["column"].unique():
    for code in X[col].unique():
        if code not in code_table["code"].values:
            missing.append((col, code))
print("빠진 코드:", missing if missing else "없음")

pd.set_option("display.max_rows", 100)
code_table

빠진 코드: 없음


,column,code,meaning_en,뜻,count
0,checking_status,A11,... < 0 DM,잔고 0 DM 미만 (마이너스),274
1,checking_status,A12,0 <= ... < 200 DM,잔고 0 이상 200 DM 미만,269
2,checking_status,A13,... >= 200 DM / salary assignments for at leas...,잔고 200 DM 이상 또는 1년 이상 급여 이체,63
3,checking_status,A14,no checking account,당좌 계좌 없음,394
4,credit_history,A30,no credits taken / all credits paid back duly,대출 없음 / 모두 제때 상환,40
5,credit_history,A31,all credits at this bank paid back duly,이 은행 대출 모두 제때 상환,49
6,credit_history,A32,existing credits paid back duly till now,현재 대출을 지금까지 제때 상환 중,530
7,credit_history,A33,delay in paying off in the past,과거 상환 연체 있음,88
8,credit_history,A34,critical account / other credits existing (not...,위험 계좌 / 다른 은행에 대출 있음,293
9,purpose,A40,car (new),새 차,234


## 3단계: 타깃을 1 = good, 0 = bad로 바꾸기

**먼저 원본 코드 확인.** 바꾸기 전에 원본이 무엇인지 두 군데서 확인한다.
1. 설명표(`german.variables`)의 `class` 줄 → `1 = Good, 2 = Bad`
2. 실제 값 → `1`이 700명, `2`가 300명

**바꾸는 규칙**
| 원본 | 뜻 | 새 값 |
|---|---|---|
| 1 | Good (갚을 사람) | **1** = 승인해도 됨 |
| 2 | Bad (못 갚을 위험) | **0** |

- 왜 바꾸나? 나중에 "모델이 1이라고 예측 = 승인"으로 읽으면 승인율을 계산하기 쉽다. 1의 평균이 곧 good 비율(승인율)이 된다.
- `y`는 지금 컬럼이 1개인 표(DataFrame)라서, `y["class"]`로 한 줄(Series)만 꺼낸 뒤 `.map({1: 1, 2: 0})`으로 바꾼다.
- 확인: 바꾼 뒤 `1`이 700, `0`이 300이어야 하고, 매핑 안 된 값(NaN)이 0개여야 한다.

In [5]:
# 1) 원본 코드 확인
print("설명표:", german.variables.loc[german.variables["name"] == "class", "description"].values[0])
print("원본 값별 인원수:")
print(y["class"].value_counts())
print()

# 2) 1 = good → 1,  2 = bad → 0
target = y["class"].map({1: 1, 2: 0})
target.name = "good"

# 3) 확인
print("바꾼 뒤 값별 인원수:")
print(target.value_counts())
print("매핑 안 된 값(NaN) 개수:", target.isna().sum())   # 0 이어야 함
print("good 비율:", target.mean())                      # 0.7 이어야 함

설명표: 1 = Good, 2 = Bad
원본 값별 인원수:
class
1    700
2    300
Name: count, dtype: int64

바꾼 뒤 값별 인원수:
good
1    700
0    300
Name: count, dtype: int64
매핑 안 된 값(NaN) 개수: 0
good 비율: 0.7


## 4단계: 25세 미만 / 25세 이상 그룹 만들기

공정성을 비교할 두 그룹을 나눈다.
- `age < 25` → **"under 25"** (25세 미만)
- `age >= 25` → **"25+"** (25세 이상)

**왜 25세 기준인가?** 이 데이터로 한 공정성 연구와 도구들이 25세 근처를 기준으로 젊은 신청자를 따로 본다. 기준이 같아야 기존 결과와 비교할 수 있다.
> 주의: IBM의 공정성 도구 AIF360은 "25세 **초과**"를 우대 그룹으로 정의한다. 그래서 정확히 25세인 사람을 어느 쪽에 넣는지가 우리와 다르다. 우리는 대회 프로젝트 설명대로 **25세 미만**을 기준으로 쓴다.

**코드 설명**
- `X["age"] < 25`는 True/False 줄을 만든다.
- 이 줄을 `.map({True: "under 25", False: "25+"})`로 글자 이름으로 바꿔 `group`에 저장한다.
- `pd.DataFrame({"group": group, "good": target})`로 두 줄을 붙인다.
- `groupby("group")`로 그룹별로 묶고 `"good"`에 대해 계산한다.
  - `count` = 인원수
  - `mean` = good 비율 (good이 1이니까 평균 = 비율)

**확인할 것:** 두 그룹 인원수를 더하면 1,000명이어야 한다.

이 표는 **실제 정답(데이터에 기록된 결과)** 기준이고, 모델과는 아직 상관없다. 만약 25세 미만의 good 비율이 원래부터 낮다면, 모델이 그 차이를 그대로 배울 수 있다. 이 점을 기억해 두자.

In [6]:
group = (X["age"] < 25).map({True: "under 25", False: "25+"})
group.name = "group"

df = pd.DataFrame({"group": group, "good": target})

group_table = df.groupby("group")["good"].agg(["count", "mean"])
group_table.columns = ["인원수", "good 비율"]
print("인원수 합계:", group_table["인원수"].sum())   # 1000 이어야 함
print("good 비율 차이 (25+ − under 25):",
      round(group_table.loc["25+", "good 비율"] - group_table.loc["under 25", "good 비율"], 3))

group_table

인원수 합계: 1000
good 비율 차이 (25+ − under 25): 0.129


,인원수,good 비율
group,,
25+,851,0.719154
under 25,149,0.590604


## 5단계: train / test 나누기 (70% / 30%)

모델이 **처음 보는 사람**에게도 잘 맞는지 확인하려면, 학습에 쓰지 않은 사람들을 따로 떼어 둬야 한다.
- **train (70%, 700명)**: 모델이 배우는 데 쓴다.
- **test (30%, 300명)**: 마지막 평가에만 쓴다. **튜닝(설정 바꿔 보기)에는 절대 쓰지 않는다.** 테스트 점수를 보고 설정을 고치면, 테스트 세트에 맞춰진 점수가 되어 "처음 보는 사람" 평가가 아니게 된다.

**`train_test_split` 옵션**
- `test_size=0.3`: 30%를 test로 뗀다.
- `stratify=target`: good/bad 비율(70:30)이 train과 test에서 똑같이 유지되도록 나눈다. 안 그러면 우연히 한쪽에 bad가 몰릴 수 있다.
- `random_state=42`: 섞는 방식을 고정해서, 다시 실행해도 항상 같은 사람들이 나뉜다 (재현 가능).

**그룹 정보도 같이 나누기**
- `group`은 나누지 않았으니, `group.loc[X_test.index]`로 **test에 들어간 사람들의 그룹**만 꺼낸다.
- `.index` = 각 사람의 원래 행 번호라서, 같은 사람끼리 정확히 맞춰진다.

**확인할 것**
- 크기: train 700, test 300
- good 비율: 둘 다 0.7
- test 안의 25세 미만 인원: 대략 45명 (149명의 약 30%). stratify는 good/bad 기준이라 나이 그룹 비율은 정확히 맞춰지지 않는다.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# test에 들어간 사람들의 나이 그룹 (같은 행 번호로 꺼냄)
group_test = group.loc[X_test.index]

print("train 크기:", X_train.shape, " test 크기:", X_test.shape)
print("train good 비율:", round(y_train.mean(), 3), " test good 비율:", round(y_test.mean(), 3))
print()
print("test 세트 그룹별 인원수:")
print(group_test.value_counts())

train 크기: (700, 20)  test 크기: (300, 20)
train good 비율: 0.7  test good 비율: 0.7

test 세트 그룹별 인원수:
group
25+         253
under 25     47
Name: count, dtype: int64


## 6단계: 베이스라인 — 전원 승인

모델을 만들기 전에 **"아무 생각 없이 모두 승인"**하면 점수가 몇 점인지 먼저 잰다. 이게 **베이스라인(기준선)**이다.
- 모델이 이 점수보다 못하면, 모델을 쓸 이유가 없다.

**accuracy(정확도)** = 예측이 정답과 맞은 비율
- 전원 승인 = 모든 사람에게 1(good)이라고 예측
- 그러면 실제 good인 사람만 맞히니까, accuracy = test의 good 비율 = **0.7**이 나와야 한다.
- 즉 이 데이터에서는 "70%는 그냥 거저 맞힌다". 나중에 모델 accuracy가 0.7보다 얼마나 높은지를 봐야 한다.

**그룹별 승인율**
- 승인율 = 그룹 안에서 예측이 1(승인)인 비율 = 예측값의 평균
- 전원 승인이니까 두 그룹 모두 승인율 1.0, **gap(차이) = 0**
- 승인율 차이로만 보면 공정하지만, bad인 사람도 다 승인해서 쓸모는 없다. 정확도와 공정성은 따로 봐야 한다는 걸 보여주는 예시다.

**코드 설명**
- `[1] * len(y_test)`: 1을 test 인원수(300)만큼 반복한 리스트 → 전원 승인 예측
- `pd.Series(..., index=X_test.index)`: 예측에 test 사람들의 행 번호를 붙여서 `group_test`와 맞출 수 있게 한다
- `accuracy_score(정답, 예측)`: scikit-learn의 정확도 계산 함수
- `.groupby(group_test).mean()`: 그룹별 예측 평균 = 그룹별 승인율
- 결과는 `baseline_acc`, `baseline_rates`, `baseline_gap`에 저장해 두고 8단계 비교표에서 다시 쓴다.

In [ ]:
from sklearn.metrics import accuracy_score

# 전원 승인: 300명 모두 1(승인)로 예측
baseline_pred = pd.Series([1] * len(y_test), index=X_test.index)

baseline_acc = accuracy_score(y_test, baseline_pred)

# 그룹별 승인율 = 그룹별 예측 평균
baseline_rates = baseline_pred.groupby(group_test).mean()
baseline_gap = baseline_rates["25+"] - baseline_rates["under 25"]

print("베이스라인 accuracy:", round(baseline_acc, 3))   # 0.7 이어야 함
print()
print("그룹별 승인율:")
print(baseline_rates)
print()
print("gap (25+ − under 25):", round(baseline_gap, 3))   # 0 이어야 함